# Merging and Organizing

## Init

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

## Reading from Silver Schema

In [0]:
total_pitch_df = spark.table('pitch_data_2025.silver.total_pitch_data')
attack_zone_df = spark.table('pitch_data_2025.silver.attack_zone_data')
pybaseball_df = spark.table('pitch_data_2025.silver.pybaseball_data')

## Check for Duplicates

In [0]:
# Keys that will be used to join tables
keys = ['game_date', 'pitcher', 
        'player_name', 'pitch_name', 
        'release_speed', 'release_spin_rate',
        'zone', 'inning', 
        'inning_topbot', 'balls', 'strikes']

total_pitch_df.groupBy(keys).count().filter("count > 1").show()

# The same result for all three dfs
# Two duplicates out of 700,000


### Handling Duplicates

In [0]:
# Using Window and the row_number() function to assign duplicated rows a unique id

# Creating Windows
total_window = (
    Window
    .partitionBy(*keys)
    .orderBy("total_pitch_count")
)

az_py_window = (
    Window
    .partitionBy(*keys)
    .orderBy("at_bat_number", "pitch_number")
)

# Creating unique_id columns
total_pitch_df = total_pitch_df.withColumn("unique_id", F.row_number().over(total_window))
attack_zone_df = attack_zone_df.withColumn("unique_id", F.row_number().over(az_py_window))
pybaseball_df = pybaseball_df.withColumn("unique_id", F.row_number().over(az_py_window))

## Merging Data

In [0]:
complete_pitch_df = (
    total_pitch_df
    .join(attack_zone_df, on=['game_date', 'pitcher', 'player_name', 'pitch_name', 'release_speed', 'release_spin_rate',
                              'zone', 'inning', 'inning_topbot', 'balls', 'strikes', 'unique_id'], how='inner')
    .join(pybaseball_df, on=['game_date', 'pitcher', 'player_name', 'pitch_name', 'release_speed', 'release_spin_rate',
                              'zone', 'inning', 'inning_topbot', 'balls', 'strikes', 'unique_id'], how='inner')
)

# Checking all rows were joined
assert (complete_pitch_df.count() == total_pitch_df.count() == 
        attack_zone_df.count() == pybaseball_df.count()), "All rows were joined or Duplicates"

### After Merge Filter and Clean

In [0]:
# Dropping unnecessary columns
complete_pitch_df = complete_pitch_df.drop('unique_id', 'at_bat_number', 'pitch_number')

# Changing important column name
complete_pitch_df = complete_pitch_df.withColumnRenamed('pitcher', 'pitcher_id')

# Ordering Columns
complete_pitch_df = complete_pitch_df.select(
    'game_date', 'pitcher_id', 'player_name', 'age_pit_legacy', 'p_throws', 'pitch_type', 'pitch_name',
    'inning', 'inning_topbot', 'balls', 'strikes', 'events', 'description', 'total_pitch_count', 'zone', 
    'attack_zone', 'release_speed', 'release_spin_rate', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z'
)

## Player Identification Data Frame

In [0]:
player_identification = complete_pitch_df.select('pitcher_id', 'player_name', 'p_throws', 'age_pit_legacy').dropDuplicates().orderBy('pitcher_id')

complete_pitch_df = complete_pitch_df.drop('player_name', 'p_throws', 'age_pit_legacy')

## Seperating By Month

In [0]:
months = {
    'march':'03',
    'april':'04',
    'may':'05',
    'june':'06',
    'july':'07',
    'august':'08',
    'september':'09'
}

for m,d in months.items():
    globals()[f'{m}_2025_pitches'] = complete_pitch_df.filter(
        F.month("game_date") == d
    )

## Writing into Gold Schema

In [0]:
# Player df
player_identification.write.mode("overwrite").saveAsTable("pitch_data_2025.gold.player_identification")

# Monthly dfs
month_dfs = {
    'march': march_2025_pitches,
    'april': april_2025_pitches,
    'may': may_2025_pitches,
    'june': june_2025_pitches,
    'july': july_2025_pitches,
    'august': august_2025_pitches,
    'september': september_2025_pitches
}

for m,df in month_dfs.items():
    df.write.mode("overwrite").saveAsTable(f"pitch_data_2025.gold.{m}_2025_pitches")